In [7]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
import nbformat



In [40]:
ticker = "USO"
risk_free_rate = 0.0446
min_volume = 1
max_expiries = 12

In [41]:
stock = yf.Ticker(ticker)

hist = stock.history(period="1d")
spot_price = hist["Close"].iloc[-1]

print(f"{ticker} spot price: {spot_price:.2f}")

expiries = stock.options[:max_expiries]

rows = []

for expiry in expiries:
    chain = stock.option_chain(expiry)

    calls = chain.calls.copy()
    calls["optionType"] = "call"
    calls["expiration"] = expiry

    puts = chain.puts.copy()
    puts["optionType"] = "put"
    puts["expiration"] = expiry

    rows.append(calls)
    rows.append(puts)

options = pd.concat(rows, ignore_index=True)

options["expiration"] = pd.to_datetime(options["expiration"])
now = pd.Timestamp.now(tz=None)

options["daysToExpiration"] = (options["expiration"] - now).dt.days
# options["timeToExpiration"] = options["daysToExpiration"] / 365

# Mid price
options["mid"] = (options["bid"] + options["ask"]) / 2

# Moneyness
# moneyness = strike / spot
# ATM is around 1.0
options["moneyness"] = options["strike"] / spot_price

# Filter bad rows
options = options[
    (options["daysToExpiration"] > 0)
    & (options["impliedVolatility"] > 0)
    & (options["impliedVolatility"] < 5)
    & (options["bid"] > 0)
    & (options["ask"] > 0)
    & (options["volume"].fillna(0) >= min_volume)
]

# Optional: focus near ATM because far OTM options can distort the surface
options = options[
    (options["moneyness"] >= 0.7)
    & (options["moneyness"] <= 1.3)
]

print(options[[
    "contractSymbol",
    "optionType",
    "expiration",
    "strike",
    "moneyness",
    "daysToExpiration",
    "bid",
    "ask",
    "mid",
    "impliedVolatility",
    "volume",
    "openInterest",
]].head())

print(f"Contracts used: {len(options)}")

x = options["moneyness"].values
y = options["daysToExpiration"].values
z = options["impliedVolatility"].values

x_grid = np.linspace(x.min(), x.max(), 60)
y_grid = np.linspace(y.min(), y.max(), 60)

X, Y = np.meshgrid(x_grid, y_grid)

Z = griddata(
    points=(x, y),
    values=z,
    xi=(X, Y),
    method="linear"
)

# Fill missing grid points using nearest interpolation
Z_nearest = griddata(
    points=(x, y),
    values=z,
    xi=(X, Y),
    method="nearest"
)

Z = np.where(np.isnan(Z), Z_nearest, Z)

USO spot price: 142.04
        contractSymbol optionType expiration  strike  moneyness  \
39  USO260515C00100000       call 2026-05-15   100.0   0.704027   
40  USO260515C00101000       call 2026-05-15   101.0   0.711067   
41  USO260515C00102000       call 2026-05-15   102.0   0.718108   
42  USO260515C00103000       call 2026-05-15   103.0   0.725148   
43  USO260515C00104000       call 2026-05-15   104.0   0.732188   

    daysToExpiration    bid    ask     mid  impliedVolatility  volume  \
39                 1  41.80  42.45  42.125           2.132817    64.0   
40                 1  39.95  42.45  41.200           2.273442     1.0   
41                 1  38.95  41.45  40.200           2.218754     1.0   
42                 1  37.95  40.45  39.200           2.160161     2.0   
43                 1  37.00  39.25  38.125           1.914063     2.0   

    openInterest  
39         135.0  
40          16.0  
41         125.0  
42          17.0  
43          65.0  
Contracts used: 1449


In [42]:
fig = go.Figure(
    data=[
        go.Surface(
            x=X,
            y=Y,
            z=Z,
        )
    ]
)

fig.update_layout(
    title=f"{ticker} Implied Volatility Surface",
    scene=dict(
        xaxis_title="Moneyness: Strike / Spot",
        yaxis_title="Day to Expiration",
        zaxis_title="Implied Volatility",
    ),
    width=900,
    height=700,
)

fig.show()